# Data Augmentation Test

## Purpose

In this notebook, I will attempt to use Gaussian noise to generate more synthetic data to see if I can get better predictions. This is to relate to the data augmentation section of my project writeup and is a technique observed in a previous paper.

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim

import sys
from pathlib import Path
import importlib.util

here = Path.cwd().resolve()
repo_root = next((p for p in (here, *here.parents) if (p / "src" / "__init__.py").exists()), None)
if repo_root is None:
    raise RuntimeError(f"Can't find repo root from {here}")

sys.path.insert(0, str(repo_root))

# Codebase imports
from src.models.lstm import LSTM 
from src.models.masked_lstm import MaskedLSTM
from src.models.trainer import Trainer
from src.models.masked_trainer import MaskedTrainer
from src.utils.loss_functions import RMSELoss
from src.utils.config_file_parser import config_file_parser
from src.utils.plot_predictions import plot_predictions
from src.utils.plot_losses import plot_loss_curves
from src.utils.datapipeline import DataPipeline
from src.utils.window_sizes.sliding_window_pipeline import SlidingWindowPipeline, SlidingWindowBatch
from src.interpretability.pfi import permutation_feature_importance
from src.interpretability.pfi_plots import plot_pfi_radar, plot_pfi_bar
from src.utils.metric_functions import rmse

In [13]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [14]:
# Change if needed
county_name = "Fresno"
rodent_flag = True
drought_flag = True
# Timestamp to track creation of run data
timestamp = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")
config_path = Path("../../config/masked_lstm_config.ini")
if rodent_flag and drought_flag:
    data_path  = Path(f"../../data/{county_name.lower()}_agg_drought.csv")
    run_dir    = Path(f"../../data/runs/{county_name.lower()}_Rat_Drought_{timestamp}")
elif rodent_flag:
  data_path   = Path(f"../../data/merged_rodent_{county_name.lower()}_agg.csv")
  run_dir     = Path(f"../../data/runs/{county_name.lower()}_Rat_{timestamp}")
else:
    data_path  = Path(f"../../data/{county_name.lower()}_Aggregate.csv")
    run_dir    = Path(f"../../data/runs/{county_name.lower()}_noRat_{timestamp}")

# run_dir.mkdir(parents=True, exist_ok=True)
print(data_path)

../../data/fresno_agg_drought.csv


In [15]:
# load config data
lstm_params, pipeline_params = config_file_parser(config_path=config_path)
hidden_size                  = int(lstm_params["hidden_size"])
num_layers                   = int(lstm_params["num_layers"])
dropout                      = float(lstm_params["dropout"])
learning_rate                = float(lstm_params["learning_rate"])
epochs                       = int(lstm_params["epochs"])
weight_decay                 = float(lstm_params["weight_decay"])
train_frac                   = float(lstm_params["train_frac"])
test_frac                    = 1 - train_frac

# import and clean the dataframe (remove date)
df = pd.read_csv(data_path)
for col in ["YEAR_MONTH", "Year-Month", "DATE"]:
    if col in df.columns:
      df = df.drop(columns = [col])

# Isolate the features I want
feature_columns = [col for col in df.columns if col != "VFRate"]
print(feature_columns)
# Create the feature and target vectors
X = df[feature_columns].values
y = df["VFRate"].values 

['FIRE_Acres_Burned', 'PRECIP', 'WIND_EventCount', 'WIND_AvgMPH', 'WIND_RunMiles', 'AQI_PM25', 'AQI_PM10', 'EARTHQUAKE_Total', 'PESTICIDE_Total', 'POUNDS_PRODUCT_APPLIED', 'Avg_Monthly_DSCI']


In [19]:
########################################
#       Sliding Window Pipeline        #
########################################
# Create the datapipeline for sliding window calculations
pipeline = SlidingWindowPipeline(X, y, test_frac=test_frac)

# create a list of sliding window sizes from 1 to 12
sliding_window_sizes = [x for x in range(1,13)]
criterion = RMSELoss()

results = [] 

for feat_idx, feat_name in enumerate(feature_columns):
  for win_size in sliding_window_sizes:
    batch : SlidingWindowBatch = pipeline.scale(*pipeline.create_single_feature_sequences(feat_idx, win_size))
    
    model = LSTM(
      input_size = 1,
      hidden_size= hidden_size, 
      num_layers= num_layers,
      dropout= dropout
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
    trainer = Trainer(model=model, criterion=criterion, optimizer=optimizer, scaler_y = batch.scaler_y)
    
    trainer.train(
        X_train = batch.X_train.to(device), 
        y_train = batch.y_train.to(device), 
        epochs=epochs)
    preds, true = trainer.evaluate(
        batch.X_test.to(device), 
        batch.y_test.to(device)
        )
    
    rmse_sw = np.sqrt(np.mean((preds - true)**2))
    results.append({
      "feature": feat_name, 
      "window_size": win_size,
      "rmse": rmse_sw
    })
    print(f"{feat_name:20s} | w={win_size:2d} | RMSE = {rmse_sw:.4f}")

results_df = pd.DataFrame(results)
best_vals = results_df.loc[results_df.groupby("feature")["rmse"].idxmin()]


FIRE_Acres_Burned    | w= 1 | RMSE = 1.0264
FIRE_Acres_Burned    | w= 2 | RMSE = 1.4923
FIRE_Acres_Burned    | w= 3 | RMSE = 1.0503
FIRE_Acres_Burned    | w= 4 | RMSE = 1.0715
FIRE_Acres_Burned    | w= 5 | RMSE = 1.1500
FIRE_Acres_Burned    | w= 6 | RMSE = 1.4157
FIRE_Acres_Burned    | w= 7 | RMSE = 1.0496
FIRE_Acres_Burned    | w= 8 | RMSE = 0.9193
FIRE_Acres_Burned    | w= 9 | RMSE = 2.4182
FIRE_Acres_Burned    | w=10 | RMSE = 1.1161
FIRE_Acres_Burned    | w=11 | RMSE = 1.0769
FIRE_Acres_Burned    | w=12 | RMSE = 1.2691
PRECIP               | w= 1 | RMSE = 1.1185
PRECIP               | w= 2 | RMSE = 1.2279
PRECIP               | w= 3 | RMSE = 2.1404
PRECIP               | w= 4 | RMSE = 1.6507
PRECIP               | w= 5 | RMSE = 1.1276
PRECIP               | w= 6 | RMSE = 2.8683
PRECIP               | w= 7 | RMSE = 1.0699
PRECIP               | w= 8 | RMSE = 1.8672
PRECIP               | w= 9 | RMSE = 1.2981
PRECIP               | w=10 | RMSE = 1.1325
PRECIP               | w=11 | RM

In [50]:
X_train_len = int(np.floor(train_frac * len(X)))
print(X_train_len)
X_train = X[:X_train_len]
print(X_train.shape)
X_train

69
(69, 11)


array([[1.63910000e+02, 1.80000000e-01, 0.00000000e+00, 3.66774194e+00,
        8.78258065e+01, 7.00000000e+01, 5.30000000e+01, 0.00000000e+00,
        2.30560507e+01, 7.82383500e+02, 5.15422500e+02],
       [1.73000000e+01, 1.49000000e+00, 0.00000000e+00, 3.10666667e+00,
        7.44900000e+01, 9.55000000e+01, 3.85000000e+01, 0.00000000e+00,
        5.19322800e-01, 5.15678000e+02, 5.01690000e+02],
       [0.00000000e+00, 1.19000000e+00, 0.00000000e+00, 3.30645161e+00,
        7.93129032e+01, 9.40000000e+01, 1.85000000e+01, 0.00000000e+00,
        0.00000000e+00, 2.38767500e+02, 5.01530000e+02],
       [1.00000000e+00, 1.25000000e+00, 0.00000000e+00, 2.58709677e+00,
        6.20193548e+01, 1.02000000e+02, 3.30000000e+01, 0.00000000e+00,
        2.46304000e+01, 3.95196900e+02, 5.02295000e+02],
       [2.80000000e+01, 2.33000000e+00, 0.00000000e+00, 3.92857143e+00,
        9.40964286e+01, 5.15000000e+01, 1.30000000e+01, 0.00000000e+00,
        7.15428855e+01, 3.75128700e+02, 5.01995000e+